In [29]:
#Redirecting to Project Base

import os 
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent.parent
sys.path.insert(0, str(BASE))

In [30]:
## Import
from src.data import Data
from statsmodels.tsa.api import VAR
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import src.metrics as metrics

In [4]:
## Get the train data.

data = Data()
prod_id_list = data.get_prod_id_list()
NUM_DATA = 100

prod_id = prod_id_list[0]

d_train = data.get_train_data(prod_id, 'train', 'one_prod_in_all_shops').T
d_val = data.get_train_data(prod_id, 'validation', 'one_prod_in_all_shops').T



Data found in 'data' folder
Loading data...
Data loaded succesfully!


In [5]:
model = VAR(d_train)
results = model.fit()

d:\venv\time-series-venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:559: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  _index = to_datetime(index)
d:\venv\time-series-venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


In [6]:
d_train

,0,3049,6098,9147,12196,15245,18294,21343,24392,27441
d_896,0,0,0,0,0,0,0,0,0,0
d_897,0,0,0,0,0,0,0,0,0,0
d_898,0,0,0,0,0,0,0,0,0,0
d_899,0,0,0,0,0,0,0,0,0,0
d_900,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
d_1796,2,0,0,3,0,1,1,0,0,0
d_1797,2,0,0,1,1,1,5,0,0,0
d_1798,1,0,1,0,0,0,0,3,0,1
d_1799,0,0,3,0,0,0,0,0,0,1


In [7]:
lag_order = results.k_ar
print("Selected lag:", lag_order)

forecast_horizon = len(d_val) - len(d_train)
print("Forecast Horizon: ", forecast_horizon)
forecast = results.forecast(d_train.values[-lag_order:], steps=forecast_horizon)

# Convert forecast to DataFrame
d_pred = pd.DataFrame(forecast)

d_actual = d_val.iloc[-forecast_horizon:, :]

print(d_pred, d_actual)

Selected lag: 1
Forecast Horizon:  113
            0         1         2         3         4         5         6  \
0    0.535568  0.426683  0.789387  0.556138  0.218812  0.396808  0.269836   
1    0.555637  0.511548  0.715803  0.599647  0.252980  0.375895  0.358156   
2    0.543945  0.507384  0.716973  0.600785  0.251669  0.386705  0.365383   
3    0.544349  0.509346  0.716933  0.600583  0.252218  0.387046  0.366862   
4    0.544264  0.509882  0.716969  0.600704  0.252181  0.387184  0.367210   
..        ...       ...       ...       ...       ...       ...       ...   
108  0.544234  0.510000  0.716960  0.600710  0.252175  0.387223  0.367316   
109  0.544234  0.510000  0.716960  0.600710  0.252175  0.387223  0.367316   
110  0.544234  0.510000  0.716960  0.600710  0.252175  0.387223  0.367316   
111  0.544234  0.510000  0.716960  0.600710  0.252175  0.387223  0.367316   
112  0.544234  0.510000  0.716960  0.600710  0.252175  0.387223  0.367316   

            7         8         9  


In [33]:
## Calculate Metrics and take average for each TS:
metric_names = [
    'MAE',
    'RMSE',
    'WAPE',
    'WRMSE',
    'mean_error'
    ]

avg_scores = {m : [] for m in metric_names}
for y_pred, y_true in zip(d_pred.T.items(), d_actual.T.items()):
    y_pred = y_pred[1].to_numpy()
    y_true = y_true[1].to_numpy()
    for met in metric_names:
        # avg_scores[met].append(metrics.MAE(y_pred[1].to_numpy(), y_true[1].to_numpy()))
        avg_scores[met].append(getattr(metrics, met)(y_pred, y_true))
        # print(y_pred[1].to_numpy(), y_true[1].to_numpy())

results_df = pd.DataFrame({
    "Metric": metric_names,
    "Average Score": [
        np.mean(avg_scores[met])
        for met in metric_names
    ]
})

print(results_df.to_string(index=False))

    Metric  Average Score
       MAE       0.594353
      RMSE       0.788758
      WAPE       1.545878
     WRMSE       2.494273
mean_error       0.159721
